# Lesson 3 · Pandas Foundations: Series, DataFrames, Importing & Selecting Data
### From NumPy arrays to real, labelled tables

| | |
|---|---|
| **Type** | LESSON 03 |
| **Time** | about 120 minutes |
| **Level** | Beginner |
| **Prerequisites** | Lessons 1–2 (NumPy arrays, boolean masks) |

**By the end of this notebook you will be able to:**

- Build Series and DataFrames and explain how they differ from NumPy arrays
- Import data from **CSV, Excel and JSON** (including nested JSON and Excel sheets with title rows)
- Run the 30-second 'first look' ritual on any unfamiliar table
- Select rows and columns with `[]`, `.loc` and `.iloc`
- Filter with Boolean conditions, `isin`, `between`, `str.contains` and `query`
- Sort, rank and create new columns

## 🛠️ Setup
Run this cell first. It imports the tools and loads the data every problem below relies on.

In [1]:

import sys; sys.path.insert(0, "..")
import json
import numpy as np
import pandas as pd
from course_utils import DATA_DIR, check, first_look

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
prices = pd.read_csv(DATA_DIR / "market_prices_clean.csv")        # weekly prices (tidy teaching version)
print("Setup complete ✔  prices:", prices.shape)


Setup complete ✔  prices: (15048, 9)


## 👋 From arrays to tables

NumPy gave us fast arrays, but real data look like **spreadsheets**: columns with *names*, rows with *labels*, a mixture of numbers,
text and dates. **pandas** is NumPy with labels, plus a huge toolbox for the daily chores of an analyst.

> 🎯 **Today's mission.** Your team just received files from the field: a CSV of prices, an Excel workbook of production figures, and a JSON file
> describing the markets. By the end of this lesson you'll be able to open all three, understand them in 30 seconds, and pull out any slice you want.

Two core objects: learn these two words and you know half of pandas:

| Object | It is… | Think of it as |
|---|---|---|
| **Series** | one labelled column | a column in a spreadsheet |
| **DataFrame** | a table of Series sharing one row index | the whole sheet |

## 1 · Series: a labelled column

In [2]:
base_prices = pd.Series([48.0, 125.0, 32.0], index=["Maize", "Beans", "Kale"], name="price_kes_per_kg")
print(base_prices)
print()
print("index :", list(base_prices.index))
print("values:", base_prices.to_numpy())
print("Beans :", base_prices["Beans"])             # access by LABEL (NumPy only had positions)
print("first :", base_prices.iloc[0])              # access by POSITION

Maize     48.0
Beans    125.0
Kale      32.0
Name: price_kes_per_kg, dtype: float64

index : ['Maize', 'Beans', 'Kale']
values: [ 48. 125.  32.]
Beans : 125.0
first : 48.0


The superpower of the index is **alignment**: pandas matches values by label, not by position.

In [3]:
a = pd.Series({"Maize": 50, "Beans": 120, "Kale": 30})
b = pd.Series({"Beans": 10, "Kale": 5, "Onions": 8})
print("a + b:\n", a + b)
print("\nwith fill_value=0:\n", a.add(b, fill_value=0))

a + b:
 Beans     130.0
Kale       35.0
Maize       NaN
Onions      NaN
dtype: float64

with fill_value=0:
 Beans     130.0
Kale       35.0
Maize      50.0
Onions      8.0
dtype: float64


> 🧠 **Pause and predict.** Why is Maize `NaN` in `a + b`? (Because `b` has no Maize; 'something + nothing' is unknown, not zero. `fill_value=0` changes that rule.)

## 2 · DataFrame: a table

In [4]:
mini = pd.DataFrame({
    "crop":   ["Maize", "Beans", "Kale", "Onions"],
    "price":  [48.0, 125.0, 32.0, 72.0],
    "volume": [40000, 15000, 18000, 14000],
})
print(mini)
print("\nshape  :", mini.shape)
print("columns:", list(mini.columns))
print("index  :", mini.index)
print("dtypes :\n", mini.dtypes)

     crop  price  volume
0   Maize   48.0   40000
1   Beans  125.0   15000
2    Kale   32.0   18000
3  Onions   72.0   14000

shape  : (4, 3)
columns: ['crop', 'price', 'volume']
index  : RangeIndex(start=0, stop=4, step=1)
dtypes :
 crop          str
price     float64
volume      int64
dtype: object


> 📝 **Note.** In pandas 2.x, text columns show dtype `object`; in pandas 3 they show `str`. Same idea: a column of text.

**Columns are Series. Selecting one column gives a Series; selecting a *list* of columns gives a DataFrame.**

In [5]:
print(type(mini["price"]), "|", type(mini[["price"]]), "|", type(mini[["crop", "price"]]))
mini["revenue"] = mini["price"] * mini["volume"]            # create a column: vectorised, no loop
mini = mini.rename(columns={"revenue": "revenue_kes"})
print(mini)
print(mini.drop(columns="volume"))                          # returns a NEW table; mini is unchanged

<class 'pandas.Series'> | <class 'pandas.DataFrame'> | <class 'pandas.DataFrame'>
     crop  price  volume  revenue_kes
0   Maize   48.0   40000    1920000.0
1   Beans  125.0   15000    1875000.0
2    Kale   32.0   18000     576000.0
3  Onions   72.0   14000    1008000.0
     crop  price  revenue_kes
0   Maize   48.0    1920000.0
1   Beans  125.0    1875000.0
2    Kale   32.0     576000.0
3  Onions   72.0    1008000.0


> ⚠️ **Common mistake.** `df.price` works as a shortcut, but fails when the column name has spaces, clashes with a method (`df.count`), or when you want to *create* a column. Prefer `df['price']` in serious code.

## 3 · Importing data: CSV, Excel, JSON

Real data live in files. Here's the data folder for this course:

| File | Format | What it holds |
|---|---|---|
| `market_prices_clean.csv` | CSV | weekly prices & volumes (tidy) |
| `market_prices_raw.csv` | CSV | the same, *with real-world mess* (Lesson 4 & the project) |
| `production_by_county.xlsx` | Excel | production by county/crop/year + county facts |
| `markets.json` | nested JSON | market metadata with coordinates |
| `weather_monthly.json` | JSON records | monthly rainfall & temperature by county |
| `dairy_farms.csv` | CSV | a dairy-farm survey (Lesson 8) |

### 3.1 CSV: `pd.read_csv`

In [6]:
raw = pd.read_csv(DATA_DIR / "market_prices_raw.csv")
print(raw.shape)
raw.head()

(15093, 9)


,record_id,date,recorded_at,crop,market,county,price_kes_per_kg,price_unit,volume_kg
0,R012281,2024-04-08,2024-04-08 07:51:00,Cabbage,Machakos Market,Machakos,40.8,KES/kg,10520.0
1,R008502,2023-04-10,2023-04-10 07:46:00,Potatoes,City Market,Nairobi,64.3,KES/kg,19790.0
2,R014632,2024-11-25,2024-11-25 09:35:00,Beans,Karatina Market,Nyeri,168.1,KES/kg,14540.0
3,R014555,2024-11-18,2024-11-18 07:55:00,Maize,Eldoret Municipal Market,Uasin Gishu,50.9,KES/kg,34110.0
4,R012442,2024-04-22,2024-04-22 09:13:00,Tomatoes,Nakuru Municipal Market,Nakuru,90.5,KES/kg,21370.0


`read_csv` has dozens of options. These six cover 90% of real life:

| Option | Use it to… |
|---|---|
| `usecols=[...]` | load only some columns (saves memory) |
| `nrows=1000` | peek at the first 1000 rows of a giant file |
| `dtype={"county": "category"}` | force types (IDs as text, labels as category) |
| `parse_dates=["date"]` | turn text into real dates on the way in |
| `na_values=["N/A", "-", ""]` | tell pandas what *else* means missing |
| `sep=";"`, `encoding="latin-1"` | handle other delimiters / text encodings |

In [7]:
small = pd.read_csv(DATA_DIR / "market_prices_raw.csv",
                    usecols=["date", "crop", "county", "price_kes_per_kg"],
                    parse_dates=["date"],
                    dtype={"crop": "category"},
                    nrows=2000)
print(small.dtypes)
small.head(3)

date                datetime64[us]
crop                      category
county                         str
price_kes_per_kg           float64
dtype: object


,date,crop,county,price_kes_per_kg
0,2024-04-08,Cabbage,Machakos,40.8
1,2023-04-10,Potatoes,Nairobi,64.3
2,2024-11-25,Beans,Nyeri,168.1


### 3.2 Excel: `pd.read_excel`

Excel files can contain several **sheets**, and people love to put titles above the real header. Let's look inside.

In [8]:
xl = pd.ExcelFile(DATA_DIR / "production_by_county.xlsx")
print("Sheets:", xl.sheet_names)

oops = pd.read_excel(xl, sheet_name="production")           # naive read
print("\nNaive read -> columns:", list(oops.columns)[:3], "...  (a title row is being treated as the header!)")

production = pd.read_excel(xl, sheet_name="production", skiprows=2)   # skip the 2 title rows
counties = pd.read_excel(xl, sheet_name="counties")
print("\nFixed read -> columns:", list(production.columns))
production.head()

Sheets: ['production', 'counties']

Naive read -> columns: ['Crop production by county, 2021-2024 (SYNTHETIC teaching data)', 'Unnamed: 1', 'Unnamed: 2'] ...  (a title row is being treated as the header!)



Fixed read -> columns: ['county', 'crop', 'year', 'area_ha', 'production_tonnes']


,county,crop,year,area_ha,production_tonnes
0,Nairobi,Kale,2021,310.0,4417.6
1,Nairobi,Tomatoes,2021,146.0,2702.1
2,Nairobi,Kale,2022,289.0,3612.5
3,Nairobi,Tomatoes,2022,147.0,2094.1
4,Nairobi,Kale,2023,292.0,4467.9


> 🌍 **In the real world.** Spreadsheets made for humans contain titles, merged cells, notes and totals. Reading them is **detective work**: always look at the file first and tune `skiprows`, `header`, `usecols`, `nrows`. (Needs the `openpyxl` package installed: it is in `requirements.txt`.)

### 3.3 JSON: flat and nested

JSON is how web services talk. **Flat** JSON (a list of records) loads directly:

In [9]:
weather = pd.read_json(DATA_DIR / "weather_monthly.json")
weather.head(3)

,county,year,month,rainfall_mm,avg_temp_c
0,Nairobi,2021,1,84.9,20.3
1,Nairobi,2021,2,27.4,20.0
2,Nairobi,2021,3,126.3,20.5


**Nested** JSON (dictionaries inside dictionaries, lists inside dictionaries) needs `json.load` + `pd.json_normalize`, which flattens the nesting into columns:

In [10]:
with open(DATA_DIR / "markets.json") as f:
    markets_raw = json.load(f)
print("One raw record:\n", json.dumps(markets_raw[0], indent=2))

markets = pd.json_normalize(markets_raw)
markets.columns = markets.columns.str.replace(".", "_", regex=False)   # location.lat -> location_lat
markets.head(3)

One raw record:
 {
  "market": "Wakulima Market",
  "county": "Nairobi",
  "type": "wholesale",
  "location": {
    "lat": -1.281,
    "lon": 36.83
  },
  "opens": {
    "days": "Daily",
    "hour": 6
  },
  "crops_traded": [
    "Bananas",
    "Beans",
    "Cabbage",
    "Kale",
    "Maize",
    "Onions",
    "Potatoes",
    "Tomatoes"
  ]
}


,market,county,type,crops_traded,location_lat,location_lon,opens_days,opens_hour
0,Wakulima Market,Nairobi,wholesale,"[Bananas, Beans, Cabbage, Kale, Maize, Onions,...",-1.2810,36.8300,Daily,6
1,City Market,Nairobi,retail,"[Beans, Cabbage, Kale, Maize, Onions, Potatoes]",-1.2833,36.8219,Mon-Sat,6
2,Kongowea Market,Mombasa,wholesale,"[Bananas, Beans, Cabbage, Kale, Maize, Onions,...",-4.0000,39.6830,Daily,6


A list inside a cell (`crops_traded`) can be unpacked into one row per item with `explode`:

In [11]:
market_crop_pairs = markets[["market", "crops_traded"]].explode("crops_traded")
print(market_crop_pairs.shape, "-> one row per (market, crop) pair")
market_crop_pairs.head(4)

(72, 2) -> one row per (market, crop) pair


,market,crops_traded
0,Wakulima Market,Bananas
0,Wakulima Market,Beans
0,Wakulima Market,Cabbage
0,Wakulima Market,Kale


### 3.4 Saving your work

In [12]:
import tempfile, os
out = os.path.join(tempfile.gettempdir(), "demo_markets.csv")
markets[["market", "county", "type"]].to_csv(out, index=False)     # index=False: don't write the 0,1,2.. row numbers
print(pd.read_csv(out).head(3))

            market   county       type
0  Wakulima Market  Nairobi  wholesale
1      City Market  Nairobi     retail
2  Kongowea Market  Mombasa  wholesale


## 4 · The 30-second "first look" ritual

Whenever you meet a new table, do **the same checks in the same order**. I even gave you a helper that bundles them:

In [13]:
first_look(prices)

📐 Shape: 15,048 rows × 9 columns
🔁 Fully duplicated rows: 0


,dtype,missing,missing_%,unique
record_id,str,0,0.0,15048
date,str,0,0.0,209
recorded_at,str,0,0.0,13401
crop,str,0,0.0,8
market,str,0,0.0,10
county,str,0,0.0,9
price_kes_per_kg,float64,0,0.0,1666
price_unit,str,0,0.0,1
volume_kg,int64,0,0.0,4234


,record_id,date,recorded_at,crop,market,county,price_kes_per_kg,price_unit,volume_kg
0,R000001,2021-01-04,2021-01-04 08:58:00,Beans,City Market,Nairobi,143.5,KES/kg,12260
1,R000002,2021-01-04,2021-01-04 08:36:00,Cabbage,City Market,Nairobi,31.1,KES/kg,24020
2,R000003,2021-01-04,2021-01-04 07:56:00,Kale,City Market,Nairobi,39.1,KES/kg,13100
3,R000004,2021-01-04,2021-01-04 09:23:00,Maize,City Market,Nairobi,50.5,KES/kg,35550
4,R000005,2021-01-04,2021-01-04 07:00:00,Onions,City Market,Nairobi,90.1,KES/kg,8580


And the individual tools behind it:

In [14]:
print(prices.shape)                       # rows, columns
prices.info()                             # types + non-null counts + memory

(15048, 9)


<class 'pandas.DataFrame'>
RangeIndex: 15048 entries, 0 to 15047
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   record_id         15048 non-null  str    
 1   date              15048 non-null  str    
 2   recorded_at       15048 non-null  str    
 3   crop              15048 non-null  str    
 4   market            15048 non-null  str    
 5   county            15048 non-null  str    
 6   price_kes_per_kg  15048 non-null  float64
 7   price_unit        15048 non-null  str    
 8   volume_kg         15048 non-null  int64  
dtypes: float64(1), int64(1), str(7)
memory usage: 1.0 MB


In [15]:
prices.describe()                         # numeric summary: count, mean, std, min, quartiles, max

,price_kes_per_kg,volume_kg
count,15048.000000,15048.000000
mean,68.961709,23733.121345
std,40.221603,12364.710196
min,20.500000,4440.000000
25%,40.900000,15160.000000
50%,53.900000,20440.000000
75%,82.100000,28792.500000
max,213.700000,106750.000000


In [16]:
prices.describe(include="all").T.head(8)  # include text columns too (unique, top, freq)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
record_id,15048,15048,R000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,15048,209,2021-01-04,72,NaN,NaN,NaN,NaN,NaN,NaN,NaN
recorded_at,15048,13401,2021-06-21 10:48:00,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
crop,15048,8,Beans,2090,NaN,NaN,NaN,NaN,NaN,NaN,NaN
market,15048,10,Eldoret Municipal Market,1672,NaN,NaN,NaN,NaN,NaN,NaN,NaN
county,15048,9,Nairobi,2926,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price_kes_per_kg,15048.0,NaN,NaN,NaN,68.961709,40.221603,20.5,40.9,53.9,82.1,213.7
price_unit,15048,1,KES/kg,15048,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
print(prices["crop"].value_counts())              # frequency of each category
print()
print(prices["market"].nunique(), "markets;", prices["crop"].nunique(), "crops")
print("\nA random sample is more honest than head() when data are sorted:")
prices.sample(4, random_state=1)

crop
Beans       2090
Kale        2090
Potatoes    2090
Cabbage     1881
Maize       1881
Onions      1672
Bananas     1672
Tomatoes    1672
Name: count, dtype: int64

10 markets; 8 crops

A random sample is more honest than head() when data are sorted:


,record_id,date,recorded_at,crop,market,county,price_kes_per_kg,price_unit,volume_kg
9369,R009370,2023-07-03,2023-07-03 10:59:00,Kale,Eldoret Municipal Market,Uasin Gishu,33.2,KES/kg,17520
3532,R003533,2021-12-13,2021-12-13 06:32:00,Onions,City Market,Nairobi,105.6,KES/kg,10600
4674,R004675,2022-03-28,2022-03-28 10:05:00,Cabbage,Wakulima Market,Nairobi,34.3,KES/kg,32590
3499,R003500,2021-12-06,2021-12-06 10:28:00,Potatoes,Machakos Market,Machakos,53.2,KES/kg,24710


> 💡 **Intuition.** `head()` shows you the **best-behaved** rows (often the oldest, sorted ones). `sample()`, `tail()` and `describe()` show you what `head()` hides.

## 5 · Selecting and filtering

### 5.1 Columns

In [18]:
cols = prices[["date", "crop", "market", "price_kes_per_kg"]]
cols.head(3)

,date,crop,market,price_kes_per_kg
0,2021-01-04,Beans,City Market,143.5
1,2021-01-04,Cabbage,City Market,31.1
2,2021-01-04,Kale,City Market,39.1


### 5.2 Rows & cells: `.loc` (labels) vs `.iloc` (positions)

| | `.loc[rows, cols]` | `.iloc[rows, cols]` |
|---|---|---|
| Selects by | **labels** (index values, column names) | **integer positions** |
| Slice end | **inclusive** | exclusive |
| Accepts Boolean masks? | yes | only plain arrays |

In [19]:
print(prices.loc[0:2, ["crop", "price_kes_per_kg"]])      # labels 0,1,2  (end INCLUDED)
print()
print(prices.iloc[0:2, 0:3])                              # positions 0,1 (end EXCLUDED), columns 0,1,2

      crop  price_kes_per_kg
0    Beans             143.5
1  Cabbage              31.1
2     Kale              39.1

  record_id        date          recorded_at
0   R000001  2021-01-04  2021-01-04 08:58:00
1   R000002  2021-01-04  2021-01-04 08:36:00


With a meaningful index, `.loc` reads like English:

In [20]:
by_id = prices.set_index("record_id")
print(by_id.loc["R000100", ["crop", "market", "price_kes_per_kg"]])

crop                       Onions
market              Kibuye Market
price_kes_per_kg             71.4
Name: R000100, dtype: object


### 5.3 Boolean filtering: the bread and butter

It's exactly the mask idea from Lesson 1, now with column names.

In [21]:
maize = prices[prices["crop"] == "Maize"]
expensive_tomatoes = prices[(prices["crop"] == "Tomatoes") & (prices["price_kes_per_kg"] > 120)]
print(len(maize), "maize rows;", len(expensive_tomatoes), "very expensive tomato rows")
expensive_tomatoes[["date", "market", "price_kes_per_kg"]].head()

1881 maize rows; 10 very expensive tomato rows


,date,market,price_kes_per_kg
7589,2023-01-09,Kibuye Market,120.7
10071,2023-09-04,Thika Market,123.4
11151,2023-12-18,Thika Market,120.6
12303,2024-04-08,Thika Market,123.5
12375,2024-04-15,Thika Market,138.1


More filtering tools, each with a typical use:

In [22]:
coast_or_lake = prices[prices["county"].isin(["Mombasa", "Kisumu"])]                      # membership
mid_price = prices[prices["price_kes_per_kg"].between(40, 50)]                            # inclusive range
municipal = prices[prices["market"].str.contains("Municipal")]                            # text search
print(len(coast_or_lake), len(mid_price), len(municipal))

# query(): the same filters, readable as a sentence
threshold = 120
print(len(prices.query("crop == 'Beans' and price_kes_per_kg > @threshold")), "bean records above", threshold)

3344 3075 4598
2045 bean records above 120


> ⚠️ **Common mistake.** **Chained assignment.** `prices[prices['crop']=='Maize']['price_kes_per_kg'] = 0` *looks* like it edits the table but modifies a temporary copy (pandas 2 warns, pandas 3 silently ignores). Always assign with one `.loc`: `prices.loc[prices['crop']=='Maize', 'price_kes_per_kg'] = 0`.

### 5.4 Sorting and ranking

In [23]:
top = prices.sort_values("price_kes_per_kg", ascending=False).head(3)
print(top[["date", "crop", "market", "price_kes_per_kg"]])

multi = prices.sort_values(["crop", "price_kes_per_kg"], ascending=[True, False])      # A→Z by crop, then dearest first
print(multi[["crop", "price_kes_per_kg"]].head(3))

print(prices.nlargest(3, "volume_kg")[["crop", "market", "volume_kg"]])               # shortcut for "top n"

             date   crop           market  price_kes_per_kg
12423  2024-04-22  Beans  Machakos Market             213.7
12495  2024-04-29  Beans  Machakos Market             212.8
12567  2024-05-06  Beans  Machakos Market             205.6


          crop  price_kes_per_kg
14078  Bananas              69.2
12356  Bananas              67.8
12644  Bananas              66.9


        crop           market  volume_kg
572    Maize  Wakulima Market     106750
11156  Maize  Wakulima Market     104870
11228  Maize  Wakulima Market     102520


### 5.5 New columns and `assign`

In [24]:
tidy = prices.assign(
    revenue_kes=lambda d: d["price_kes_per_kg"] * d["volume_kg"],
    price_usd=lambda d: (d["price_kes_per_kg"] / 129).round(3),
)
tidy[["crop", "price_kes_per_kg", "price_usd", "revenue_kes"]].head(3)

,crop,price_kes_per_kg,price_usd,revenue_kes
0,Beans,143.5,1.112,1759310.0
1,Cabbage,31.1,0.241,747022.0
2,Kale,39.1,0.303,512210.0


> 💡 **Intuition.** `assign` (with `lambda d: ...`) lets you *chain* steps: each step can use columns created by the previous one, and the original table is never modified.

---
# 🧑‍🏫 Worked examples

## Worked example 1: "When and where was tomato at its dearest?"

> **Thinking steps.** (1) Restrict to tomatoes. (2) Find the **label** of the maximum with `idxmax()`. (3) Use `.loc` with that label to read the whole row.

In [25]:
tom = prices[prices["crop"] == "Tomatoes"]
i = tom["price_kes_per_kg"].idxmax()
print(prices.loc[i, ["date", "market", "county", "price_kes_per_kg"]])

date                  2024-04-15
market              Thika Market
county                    Kiambu
price_kes_per_kg           138.1
Name: 12375, dtype: object


## Worked example 2: "Top 5 revenue weeks for maize at Wakulima"

> **Thinking steps.** (1) Filter two conditions. (2) Create `revenue`. (3) `nlargest(5, ...)`. (4) Show a tidy subset of columns.

In [26]:
w = prices[(prices["crop"] == "Maize") & (prices["market"] == "Wakulima Market")]
w = w.assign(revenue_kes=w["price_kes_per_kg"] * w["volume_kg"])
w.nlargest(5, "revenue_kes")[["date", "price_kes_per_kg", "volume_kg", "revenue_kes"]]

,date,price_kes_per_kg,volume_kg,revenue_kes
5828,2022-07-18,72.4,96460,6983704.0
12452,2024-04-22,66.8,100880,6738784.0
7556,2023-01-02,72.8,89490,6514872.0
7916,2023-02-06,74.0,83530,6181220.0
14900,2024-12-16,59.7,98950,5907315.0


## Worked example 3: "Which crops does a county grow?" (Excel)

> **Thinking steps.** (1) Read the correct sheet with `skiprows`. (2) Filter on county. (3) Sort by production.

In [27]:
prod = pd.read_excel(DATA_DIR / "production_by_county.xlsx", sheet_name="production", skiprows=2)
nakuru_2023 = prod[(prod["county"] == "Nakuru") & (prod["year"] == 2023)]
nakuru_2023.sort_values("production_tonnes", ascending=False)

,county,crop,year,area_ha,production_tonnes
64,Nakuru,Potatoes,2023,21371.0,283864.4
62,Nakuru,Maize,2023,73089.0,191366.1
65,Nakuru,Cabbage,2023,5191.0,147213.2
66,Nakuru,Kale,2023,3581.0,57347.9
67,Nakuru,Tomatoes,2023,2647.0,57216.9
63,Nakuru,Beans,2023,26349.0,26084.7
68,Nakuru,Onions,2023,1482.0,22604.9


---
# 🏋️ Problems

Variables available from Setup: `prices` (clean weekly data), `pd`, `np`, `json`, `DATA_DIR`.

### 🧩 Problem 3.1: How many Mombasa tomato records?

Count the rows in `prices` where the crop is **Tomatoes** and the county is **Mombasa**.

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

Two conditions, joined with `&`, each in its own parentheses.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Sum the resulting Boolean Series, or take the `len` of the filtered table.

</details>

*Auto-check id:* `03-p1`  ·  *Set the variable `answer`, then run the cell.*

In [28]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p1', answer)

⏳ 03-p1: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 3.2: Average bean price at Kibuye

What is the mean `price_kes_per_kg` of **Beans** at **Kibuye Market**? Round to 2 decimals.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

Filter first (two conditions), then pick the price column and call `.mean()`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`prices.loc[mask, 'price_kes_per_kg'].mean()` is a neat one-liner once you have the mask.

</details>

*Auto-check id:* `03-p2`  ·  *Set the variable `answer`, then run the cell.*

In [29]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p2', answer)

⏳ 03-p2: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 3.3: Where was onion at its dearest?

Which **market** recorded the single highest onion price in `prices`?

*Expected type:* a string (the market name).

<details>
<summary>💡 Hint 1</summary>

Filter to onions, find the label of the highest price with `idxmax()`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Then use `prices.loc[label, 'market']`.

</details>

*Auto-check id:* `03-p3`  ·  *Set the variable `answer`, then run the cell.*

In [30]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p3', answer)

⏳ 03-p3: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 3.4: What does Thika sell?

Return the **alphabetically sorted list** of crops sold at **Thika Market**.

*Expected type:* a list of strings.

<details>
<summary>💡 Hint 1</summary>

`.unique()` gives the distinct values of a column.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Wrap with `sorted(...)`; convert to a plain list if you like.

</details>

*Auto-check id:* `03-p4`  ·  *Set the variable `answer`, then run the cell.*

In [31]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p4', answer)

⏳ 03-p4: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 3.5: Maize production in 2023 (Excel)

Read the **production** sheet of `production_by_county.xlsx` (remember the title rows!). What was the **total maize production (tonnes) in 2023** across all counties? Round to 1 decimal.

*Expected type:* float.

<details>
<summary>💡 Hint 1</summary>

Use `pd.read_excel(DATA_DIR / 'production_by_county.xlsx', sheet_name='production', skiprows=2)`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Filter crop == 'Maize' and year == 2023, then sum `production_tonnes`.

</details>

*Auto-check id:* `03-p5`  ·  *Set the variable `answer`, then run the cell.*

In [32]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p5', answer)

⏳ 03-p5: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 3.6: Markets south of the equator (nested JSON)

Load `markets.json` with `json.load`, flatten it with `pd.json_normalize`, and count how many markets have **latitude below 0** (i.e. lie south of the equator).

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

Open the file with `open(DATA_DIR / 'markets.json')` and call `json.load(f)`.

</details>

<details>
<summary>💡 Hint 2</summary>

After `json_normalize`, the nested `location` dictionary becomes columns named `location.lat` and `location.lon`.

</details>

<details>
<summary>💡 Hint 3 (the big nudge)</summary>

`(m['location.lat'] < 0).sum()`

</details>

*Auto-check id:* `03-p6`  ·  *Set the variable `answer`, then run the cell.*

In [33]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p6', answer)

⏳ 03-p6: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 3.7: The biggest revenue record

Create a revenue figure `price_kes_per_kg * volume_kg` for every row of `prices` (do **not** overwrite the table's existing columns). Return the **`record_id`** of the row with the highest revenue.

*Expected type:* a string such as `'R000123'`.

<details>
<summary>💡 Hint 1</summary>

A new Series can be created without touching `prices`: `rev = prices['price_kes_per_kg'] * prices['volume_kg']`.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

`rev.idxmax()` gives a *label*; use it with `.loc` to read the `record_id`.

</details>

*Auto-check id:* `03-p7`  ·  *Set the variable `answer`, then run the cell.*

In [34]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p7', answer)

⏳ 03-p7: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

### 🧩 Problem 3.8: Mid-priced potatoes

How many **Potatoes** records have a price **between 40 and 50 KES/kg inclusive**?

*Expected type:* integer.

<details>
<summary>💡 Hint 1</summary>

`Series.between(lo, hi)` is inclusive on both ends by default.

</details>

<details>
<summary>💡 Hint 2 (the big nudge)</summary>

Combine with the potato filter using `&`.

</details>

*Auto-check id:* `03-p8`  ·  *Set the variable `answer`, then run the cell.*

In [35]:
# ✍️ Your turn: compute the result and store it in `answer`
answer = None

check('03-p8', answer)

⏳ 03-p8: you haven't set `answer` yet. Read the problem, try it, then run this cell again.


False

## ✅ Check your understanding

Answer in your head (or out loud!) first, *then* open the answer.

**Q1. What's the difference between `df['price']` and `df[['price']]`?**

<details>
<summary>Show answer</summary>

The first returns a **Series**; the second (a list inside the brackets) returns a one-column **DataFrame**.

</details>

**Q2. `df.loc[0:3]` vs `df.iloc[0:3]`: how many rows does each return (default index)?**

<details>
<summary>Show answer</summary>

`loc` slices by label and includes the end: **4 rows** (0,1,2,3). `iloc` slices by position and excludes the end: **3 rows**.

</details>

**Q3. You read an Excel sheet and the columns are called `Unnamed: 0, Unnamed: 1…`. What probably happened?**

<details>
<summary>Show answer</summary>

There are title/blank rows above the real header. Use `skiprows=` (or `header=`) to point pandas at the right row.

</details>

**Q4. Why prefer `df.loc[mask, 'col'] = value` over `df[mask]['col'] = value`?**

<details>
<summary>Show answer</summary>

The second creates a temporary copy, so the original table isn't changed. `.loc` with both the row mask and column does a single, correct assignment.

</details>

**Q5. What does `pd.json_normalize` do?**

<details>
<summary>Show answer</summary>

It flattens nested dictionaries into columns (`location.lat`, `location.lon`); lists inside cells can then be expanded with `explode`.

</details>

## 🧾 Cheat-sheet

| I want to… | Code |
|---|---|
| Read CSV / Excel / JSON | `pd.read_csv(path)`, `pd.read_excel(path, sheet_name=, skiprows=)`, `pd.read_json(path)` |
| Flatten nested JSON | `pd.json_normalize(data)` |
| One-list → many rows | `df.explode("col")` |
| First look | `df.shape`, `df.info()`, `df.describe()`, `df.head()`, `df.sample(5)`, `df["c"].value_counts()` |
| Columns | `df["c"]`, `df[["a","b"]]`, `df.drop(columns=..)`, `df.rename(columns={..})` |
| Rows by label / position | `df.loc[rows, cols]`, `df.iloc[rows, cols]` |
| Filter | `df[df.c > 5]`, `df[(a) & (b)]`, `.isin([...])`, `.between(a,b)`, `.str.contains("x")`, `df.query("c > 5")` |
| Sort / top n | `df.sort_values(["a","b"], ascending=[True,False])`, `df.nlargest(3,"c")` |
| New column | `df["new"] = ...` or `df.assign(new=lambda d: ...)` |
| Save | `df.to_csv("out.csv", index=False)` |

## ➡️ Next up
**Lesson 4; Cleaning.** Our `raw` file has missing values, duplicates, mis-typed counties and impossible prices. Time to become a data detective.